In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import IntegerType, FloatType, DecimalType, BooleanType
from snowflake.snowpark.context import get_active_session
#Obteniendo la sesión activa
session = get_active_session()
#Definiendo el contexto
session.use_database("RETAIL_TRANSACTIONS")
session.use_schema("BRONZE")

In [ ]:
# FIJANDO EL PATH DEL RAW DATA
file_path = "@RETAIL_TRANSACTIONS.BRONZE.STG_RAW_DATA/Retail_Transaction_Dataset.csv"

# FIJANDO EL DATAFRAME INICIAL
retail_columns = [
    "CUSTOMER_ID", "PRODUCT_ID", "QUANTITY", "PRICE", "TRANSACTION_DATE", 
    "PAYMENT_METHOD", "STORE_LOCATION", "PRODUCT_CATEGORY", "DISCOUNT_APPLIED", "TOTAL_AMOUNT"
]

df_bronze = session.read.options({
    "field_delimiter": ",",
    "skip_header": 1,
    "FIELD_OPTIONALLY_ENCLOSED_BY": '"'
}).csv(file_path).to_df(retail_columns)

# GENERANDO LA TABLA BRONZE
df_bronze.write.mode("overwrite").save_as_table("RETAIL_TRANSACTIONS.BRONZE.RAW_DATA_PY")

print(f"Capa Bronze completada con exito. Se cargaron {df_bronze.count()} registros.")
df_bronze.show(10)

In [ ]:
# CARGANDO DATOS DESDE BRONZE
df_silver = session.table("RETAIL_TRANSACTIONS.BRONZE.RAW_DATA_PY")

# APLICANDO TRANSFORMACIONES
df_silver_clean = df_silver.select(
    F.cast(F.trim(F.col("CUSTOMER_ID")), IntegerType()).alias("CUSTOMER_ID"),
    F.upper(F.trim(F.col("PRODUCT_ID"))).alias("PRODUCT_ID"),
    F.cast(F.trim(F.col("QUANTITY")), IntegerType()).alias("QUANTITY"),
    F.cast(F.col("PRICE"), FloatType()).alias("PRICE"),
    F.to_timestamp(F.col("TRANSACTION_DATE"), F.lit('MM/DD/YYYY HH24:MI')).alias("TRANSACTION_TIMESTAMP"),
    F.coalesce(F.upper(F.trim(F.col("PAYMENT_METHOD"))), F.lit('UNKNOWN')).alias("PAYMENT_METHOD"),
    F.trim(F.col("STORE_LOCATION")).alias("FULL_ADDRESS"),
    # Envolvemos los parámetros en F.lit() para evitar el TypeError
    F.coalesce(
        F.regexp_substr(
            F.col("STORE_LOCATION"), 
            F.lit(r' ([A-Z]{2}) [0-9]{5}$'), 
            F.lit(1), 
            F.lit(1), 
            F.lit('e')
        ), 
        F.lit('UNKNOWN')
    ).alias("STORE_STATE"),
    F.upper(F.trim(F.col("PRODUCT_CATEGORY"))).alias("PRODUCT_CATEGORY"),
    F.coalesce(F.cast(F.col("DISCOUNT_APPLIED"), FloatType()), F.lit(0.0)).alias("DISCOUNT_PERCENT"),
    F.cast(F.col("TOTAL_AMOUNT"), FloatType()).alias("TOTAL_AMOUNT"),
    F.current_timestamp().alias("LOAD_TIMESTAMP")
).filter(
    (F.col("CUSTOMER_ID").is_not_null()) & 
    (F.col("TRANSACTION_TIMESTAMP").is_not_null()) & 
    (F.col("TOTAL_AMOUNT").is_not_null())
)

# GUARDANDO LA TABLA TRANSFORMADA
df_silver_clean.write.mode("overwrite").save_as_table("RETAIL_TRANSACTIONS.SILVER.CLEANED_RETAIL_DATA_PY")
print(f"Capa Silver completada con exito. Se transformaron {df_silver_clean.count()} registros.")

df_silver_clean.show(10)

In [ ]:
# CARGANDO LOS DATOS DESDE SILVER
df_gold = session.table("RETAIL_TRANSACTIONS.SILVER.CLEANED_RETAIL_DATA_PY")

#INSIGHT 1: RENTABILIDAD POR CATEGORIA
df_gold.group_by("PRODUCT_CATEGORY").agg([
    F.sum("TOTAL_AMOUNT").alias("TOTAL_REVENUE"),
    F.avg("DISCOUNT_PERCENT").alias("AVG_DISCOUNT_PERCENT")
]).sort(F.col("PRODUCT_CATEGORY").desc()
).create_or_replace_view("RETAIL_TRANSACTIONS.GOLD.VW_RENTABILIDAD_PORCATEGORIA_PY")

#INSIGHT 2: RENDIMIENTO GEOGRAFICO
df_gold.group_by("STORE_STATE").agg([
    F.sum("TOTAL_AMOUNT").alias("TOTAL_REVENUE"),
    F.count("*").alias("TRANSACTIONS_QTY")
]).sort(F.col("TOTAL_REVENUE").asc()
).create_or_replace_view("RETAIL_TRANSACTIONS.GOLD.VW_RENDIMIENTO_GEOGRAFICO_PY")

#INSIGHT 3: TENDENCIA VS ESTACIONALIDAD
df_gold.select(
    F.date_trunc('DAY', F.col("TRANSACTION_TIMESTAMP")).alias("TRANSACTION_DAY"),
    F.date_trunc('MONTH', F.col("TRANSACTION_TIMESTAMP")).alias("TRANSACTION_MONTH"),
    F.col("TOTAL_AMOUNT")
).group_by("TRANSACTION_DAY","TRANSACTION_MONTH").agg(
    F.sum("TOTAL_AMOUNT").alias("TOTAL_REVENUE")
).create_or_replace_view("RETAIL_TRANSACTIONS.GOLD.VW_TENDENCIA_VS_ESTACIONALIDAD_PY")

#INSIGHT 4: Análisis de Métodos de Pago
df_gold.group_by("PAYMENT_METHOD").agg([
    F.sum("TOTAL_AMOUNT").alias("TOTAL_REVENUE"),
    F.avg("TOTAL_AMOUNT").alias("AVG_TICKET_VALUE")
]).sort(F.col("TOTAL_REVENUE").desc()
).create_or_replace_view("RETAIL_TRANSACTIONS.GOLD.VW_METODO_PAGO_ANALISIS_PY")

# INSIGHT 5: Efectividad del Descuento (Lógica CASE replicada)[cite: 4]
df_gold.with_column("DISCOUNT_RANGE", 
    F.when(F.col("DISCOUNT_PERCENT") < 5, F.lit("0 - 5%"))
     .when(F.col("DISCOUNT_PERCENT") < 10, F.lit("5 - 10%"))
     .when(F.col("DISCOUNT_PERCENT") < 15, F.lit("10 - 15%"))
     .otherwise(F.lit("15 - 20%"))
).group_by("DISCOUNT_RANGE").agg(
    F.sum("QUANTITY").alias("TOTAL_UNITS_SOLD")
).sort(F.col("DISCOUNT_RANGE").asc()
).create_or_replace_view("RETAIL_TRANSACTIONS.GOLD.VW_EFECTIVIDAD_DESCUENTO_PY")

# INSIGHT 6: Ranking de Productos "Best-Sellers"[cite: 4]
df_gold.group_by("PRODUCT_ID").agg([
    F.sum("QUANTITY").alias("TOTAL_QTY_SOLD"),
    F.sum("TOTAL_AMOUNT").alias("TOTAL_REVENUE")
]).sort(F.col("TOTAL_REVENUE").desc()
).limit(10).create_or_replace_view("RETAIL_TRANSACTIONS.GOLD.VW_RANKING_PRODUCTS_PY")

# INSIGHT 7: KPI de Canasta Promedio (Average Basket Size)[cite: 4]
df_gold.group_by("PRODUCT_CATEGORY").agg(
    F.avg("QUANTITY").alias("AVG_BASKET_SIZE")
).create_or_replace_view("RETAIL_TRANSACTIONS.GOLD.VW_KPI_CANASTA_PROMEDIO_PY")

print("¡Capa Gold completada! Se crearon las 7 vistas de negocio en el esquema GOLD.")